# Credit Card Fraud Detection — Competitive Machine Learning

Kaggle Playground Series S6E11 project for detecting fraudulent credit card transactions with **CatBoost**.

**Final public Kaggle ROC-AUC: 0.94662**

This notebook documents the main analysis and modeling decisions: class imbalance, risk signals, feature interactions, numerical patterns, baseline modeling, feature importance and an ablation experiment removing the identifier.

## 1. Data loading and inspection

The training set contains **699,960 transactions** and 26 columns, including the target `is_fraud`.

The test set contains **299,984 transactions** and the same predictors without the target.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from catboost import CatBoostClassifier

DATA_PATH = "/kaggle/input/competitions/playground-series-s6e11"

train = pd.read_csv(f"{DATA_PATH}/train.csv")
test = pd.read_csv(f"{DATA_PATH}/test.csv")
sample_submission = pd.read_csv(f"{DATA_PATH}/sample_submission.csv")

print("Train:", train.shape)
print("Test:", test.shape)
display(train.head())

In [ ]:
print(train.info())
print("\nMissing values:")
print(train.isna().sum().sort_values(ascending=False).head())

print("\nColumns:")
print(train.columns.tolist())

## 2. Target imbalance

Fraud represents only **1.67%** of the training data.

This makes accuracy unsuitable as the main metric. A model that predicted every transaction as legitimate would still achieve roughly 98.3% accuracy while detecting no fraud.

The competition metric is **ROC-AUC**. **PR-AUC** is also monitored as an additional diagnostic for the rare positive class.

In [ ]:
target_counts = train["is_fraud"].value_counts()
target_pct = train["is_fraud"].value_counts(normalize=True).mul(100)

print("Quantidade:")
print(target_counts)

print("\nPercentual:")
print(target_pct)

## 3. Boolean risk signals

Several boolean variables show strong associations with fraud. The analysis below compares the fraud rate when each signal is absent or present.

In [ ]:
bool_cols = [
    "is_foreign_transaction",
    "is_new_merchant",
    "used_vpn",
    "ip_country_mismatch",
    "billing_shipping_mismatch",
    "is_ai_generated_scam_attempt"
]

for col in bool_cols:
    result = train.groupby(col)["is_fraud"].agg(["count", "mean"])
    result["fraud_pct"] = result["mean"] * 100
    print(f"\n### {col}")
    display(result)

Key findings included:

- `ip_country_mismatch=True` → ~16.5% fraud
- `is_ai_generated_scam_attempt=True` → ~15.2%
- `billing_shipping_mismatch=True` → ~13.0%
- `is_foreign_transaction=True` → ~12.6%

These values represent association, not causality.

## 4. Feature interactions

Fraud risk increases substantially when certain signals appear together. Tree-based models are particularly suitable for capturing this kind of nonlinear interaction.

In [ ]:
display(train.groupby(["used_vpn", "ip_country_mismatch"])["is_fraud"].agg(["count", "mean"]))
display(train.groupby(["is_foreign_transaction", "ip_country_mismatch"])["is_fraud"].agg(["count", "mean"]))
display(train.groupby(["billing_shipping_mismatch", "ip_country_mismatch"])["is_fraud"].agg(["count", "mean"]))

Examples observed:

- VPN + IP mismatch → ~43.7% fraud
- foreign transaction + IP mismatch → ~25.9%
- billing/shipping mismatch + IP mismatch → ~61.4%

The last pattern is extremely strong but rare, so its sample size must be considered when interpreting it.

## 5. Numerical patterns

The numerical analysis compares means and medians between legitimate and fraudulent transactions.

In [ ]:
numeric_cols = [
    "amount_usd",
    "hours_since_last_txn",
    "txn_count_last_24h",
    "distance_from_home_km",
    "card_age_months",
    "customer_age",
    "account_balance_usd",
    "cvv_retry_count",
    "velocity_score",
    "merchant_risk_score",
    "prior_disputes"
]

numeric_analysis = train.groupby("is_fraud")[numeric_cols].agg(["mean", "median"]).T
display(numeric_analysis)

In [ ]:
for col in ["cvv_retry_count", "txn_count_last_24h", "prior_disputes"]:
    result = train.groupby(col)["is_fraud"].agg(["count", "mean"])
    result["fraud_pct"] = result["mean"] * 100
    print(f"\n### {col}")
    display(result)

Important patterns:

- higher `merchant_risk_score` in fraudulent transactions;
- higher `velocity_score`;
- sharp increase in fraud rate as `cvv_retry_count` increases;
- more transactions in the previous 24 hours;
- shorter time since the previous transaction.

For CVV retries, the fraud rate rises from roughly **0.93%** at 0 retries to **41.73%** at 2 retries.

## 6. CatBoost baseline

The baseline uses the original competition features. The 80/20 split is stratified to preserve the fraud rate in both training and validation.

CatBoost was selected because it handles categorical variables directly and captures nonlinear relationships and interactions in tabular data.

In [ ]:
TARGET = "is_fraud"

X = train.drop(columns=[TARGET])
y = train[TARGET].astype(int)

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Treino:", X_train.shape)
print("Validação:", X_valid.shape)
print("Fraude treino:", y_train.mean())
print("Fraude validação:", y_valid.mean())

In [ ]:
model = CatBoostClassifier(
    iterations=1200,
    learning_rate=0.05,
    depth=7,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=100
)

valid_pred = model.predict_proba(X_valid)[:, 1]

roc_auc = roc_auc_score(y_valid, valid_pred)
pr_auc = average_precision_score(y_valid, valid_pred)

print(f"ROC-AUC: {roc_auc:.6f}")
print(f"PR-AUC: {pr_auc:.6f}")
print("Best iteration:", model.get_best_iteration())

### Baseline result

- ROC-AUC: **0.944222**
- PR-AUC: **0.509792**
- best iteration: **930**

The PR-AUC is particularly informative because the positive class represents only 1.67% of the dataset.

## 7. Feature importance

Feature importance is used to compare the model's behavior with the exploratory findings.

In [ ]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.get_feature_importance()
}).sort_values("importance", ascending=False)

display(feature_importance)

top_features = feature_importance.head(15)

plt.figure(figsize=(10, 6))
plt.barh(top_features["feature"][::-1], top_features["importance"][::-1])
plt.xlabel("Importância")
plt.ylabel("Feature")
plt.title("Top 15 Features - CatBoost Baseline")
plt.tight_layout()
plt.show()

Among the strongest features were `merchant_risk_score`, `velocity_score`, `customer_age`, `cvv_retry_count`, `auth_method` and `is_new_merchant`.

This is consistent with several patterns observed during EDA.

## 8. ID ablation

Because `id` is only an identifier, a controlled experiment was performed to test whether it contributes predictive signal.

The model architecture, split and hyperparameters remain unchanged. The only difference is the removal of `id`.

In [ ]:
X_no_id = X.drop(columns=["id"])
cat_cols_no_id = X_no_id.select_dtypes(include=["object", "category"]).columns.tolist()

X_train_2, X_valid_2, y_train_2, y_valid_2 = train_test_split(
    X_no_id,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model_no_id = CatBoostClassifier(
    iterations=1200,
    learning_rate=0.05,
    depth=7,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100
)

model_no_id.fit(
    X_train_2,
    y_train_2,
    cat_features=cat_cols_no_id,
    eval_set=(X_valid_2, y_valid_2),
    early_stopping_rounds=100
)

pred_no_id = model_no_id.predict_proba(X_valid_2)[:, 1]

roc_no_id = roc_auc_score(y_valid_2, pred_no_id)
pr_no_id = average_precision_score(y_valid_2, pred_no_id)

print(f"ROC-AUC sem ID: {roc_no_id:.6f}")
print(f"PR-AUC sem ID: {pr_no_id:.6f}")
print("Best iteration:", model_no_id.get_best_iteration())

### Ablation result

| Experiment | ROC-AUC | PR-AUC |
|---|---:|---:|
| With `id` | 0.944222 | **0.509792** |
| Without `id` | **0.944286** | 0.508453 |

The ROC-AUC improvement is only **0.000064**, meaning the identifier provides no meaningful predictive advantage. The final submission therefore uses the cleaner feature set without `id`.

## 9. Final model and Kaggle submission

The best iteration found during validation is used to retrain the final CatBoost model on the full training set. Predictions are probabilities, as required for ROC-AUC scoring.

In [ ]:
best_iterations = model_no_id.get_best_iteration() + 1

final_model = CatBoostClassifier(
    iterations=best_iterations,
    learning_rate=0.05,
    depth=7,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100
)

final_model.fit(X_no_id, y, cat_features=cat_cols_no_id)

X_test_final = test[X_no_id.columns]
test_predictions = final_model.predict_proba(X_test_final)[:, 1]

submission = sample_submission.copy()
submission["is_fraud"] = test_predictions
submission.to_csv("/kaggle/working/submission.csv", index=False)

print("Shape:", submission.shape)
print(submission.isna().sum())
display(submission.head())

## 10. Final result

The submission achieved a **Public Kaggle ROC-AUC of 0.94662**, slightly above the local validation ROC-AUC of 0.944286.

### Main takeaways

- severe class imbalance requires appropriate evaluation metrics;
- fraud risk is strongly associated with contextual and behavioral signals;
- interactions between flags can carry much more information than isolated variables;
- CatBoost performs strongly on this mixed tabular dataset;
- controlled ablation showed that `id` does not meaningfully improve performance;
- the local validation result generalized well to the public leaderboard.

### Next steps

Potential improvements:

- Stratified K-Fold and OOF predictions;
- validated feature engineering;
- LightGBM/XGBoost comparison;
- hyperparameter optimization;
- model ensembling.